# Logistic Regression — Classification the Statistical Way

**Duration:** ~2 hours
**Libraries:** NumPy · Pandas · Matplotlib · Seaborn · Scikit-learn

---

### Where we are in the ML journey

```
  Linear Regression      →   Predict a NUMBER  (exam score)
  Decision Trees         →   Predict a CATEGORY (pass/fail) using rules
  Logistic Regression    →   Predict a CATEGORY + give a PROBABILITY  ← Today
```

### What makes Logistic Regression different?

Decision Trees ask *"which box does this belong to?"*

Logistic Regression asks *"what is the **probability** this belongs to class 1?"*

> A student has a **78% chance of passing**.
> A patient has a **23% chance of having diabetes**.
> An email has a **94% chance of being spam**.

That probability output is what makes Logistic Regression the go-to model
for medical diagnosis, credit scoring, and spam detection.

---

### By the end of this notebook you will:
- Understand why we can't use Linear Regression for classification
- Know what the Sigmoid function does and why it's used
- Build and evaluate a Logistic Regression model end-to-end
- Interpret coefficients, odds ratios, and probability thresholds
- Apply everything to two datasets (student admissions + diabetes)


## Part 1 — Why Not Just Use Linear Regression?

Let's see the problem with a quick visual before writing any ML code.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay,
                              roc_curve, roc_auc_score)
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (9, 5)
np.random.seed(42)

print("✅ Libraries imported!")

In [ ]:
# Create a simple 1-feature toy dataset
hours  = np.array([1, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 7, 8, 9, 10])
passed = np.array([0, 0, 0,   0, 0,   0, 1,   0, 1,   1, 1, 1, 1, 1 ])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Linear Regression attempt ──────────────────
from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(hours.reshape(-1,1), passed)
x_line = np.linspace(0, 11, 200)
y_line = lr.predict(x_line.reshape(-1,1))

axes[0].scatter(hours, passed, color="#7c6af7", s=100, zorder=3, edgecolor="white")
axes[0].plot(x_line, y_line, color="#e06060", linewidth=2, label="Linear fit")
axes[0].axhline(0, color="white", linewidth=0.5, linestyle="--")
axes[0].axhline(1, color="white", linewidth=0.5, linestyle="--")
axes[0].fill_between(x_line, 0, 1, alpha=0.05, color="white")
axes[0].set_title("Linear Regression on Binary Data ❌", fontweight="bold")
axes[0].set_xlabel("Study Hours"); axes[0].set_ylabel("Pass (1) / Fail (0)")
axes[0].annotate("Predicts >1.0 !", xy=(10, 1.05), color="#e06060", fontsize=10)
axes[0].annotate("Predicts <0.0 !", xy=(0.2, -0.08), color="#e06060", fontsize=10)
axes[0].legend()

# ── Right: Logistic Regression ────────────────────────
from sklearn.linear_model import LogisticRegression as LR
log = LR().fit(hours.reshape(-1,1), passed)
y_prob = log.predict_proba(x_line.reshape(-1,1))[:, 1]

axes[1].scatter(hours, passed, color="#7c6af7", s=100, zorder=3, edgecolor="white")
axes[1].plot(x_line, y_prob, color="#3ecfb2", linewidth=2.5, label="Logistic curve")
axes[1].axhline(0.5, color="#f0a040", linewidth=1.5, linestyle="--", label="0.5 threshold")
axes[1].set_ylim(-0.1, 1.1)
axes[1].set_title("Logistic Regression on Binary Data ✅", fontweight="bold")
axes[1].set_xlabel("Study Hours"); axes[1].set_ylabel("Probability of Passing")
axes[1].legend()

plt.suptitle("Why Logistic Regression?", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("Linear Regression can predict values outside [0, 1] — meaningless for probability!")
print("Logistic Regression always stays between 0 and 1 — perfect for probability!")

## Part 2 — The Sigmoid Function: The Heart of Logistic Regression

Logistic Regression takes a Linear Regression output and passes it through
the **Sigmoid function**, which squashes any number into the range (0, 1).

```
         Linear output (z)         Sigmoid output σ(z)
         ─────────────────         ────────────────────
         z = w₁x₁ + w₂x₂ + b  →   σ(z) = 1 / (1 + e⁻ᶻ)
         
         z can be anything          σ(z) is always between 0 and 1
         (-∞ to +∞)                 → interpret as probability
```

The model then applies a **decision threshold** (usually 0.5):
- probability ≥ 0.5 → Class 1 (Pass / Spam / Positive)
- probability < 0.5 → Class 0 (Fail / Not Spam / Negative)


In [ ]:
# Visualise the Sigmoid function
z = np.linspace(-8, 8, 300)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(9, 4))
plt.plot(z, sigmoid, color="#3ecfb2", linewidth=2.5, label="σ(z) = 1/(1+e⁻ᶻ)")
plt.axhline(0.5, color="#f0a040", linestyle="--", linewidth=1.5, label="Threshold = 0.5")
plt.axhline(0.0, color="white",   linestyle=":",  linewidth=0.8, alpha=0.3)
plt.axhline(1.0, color="white",   linestyle=":",  linewidth=0.8, alpha=0.3)
plt.axvline(0.0, color="white",   linestyle=":",  linewidth=0.8, alpha=0.3)

# Annotations
plt.annotate("z >> 0 → σ → 1.0
(very confident: Class 1)",
             xy=(6, 0.98), fontsize=9, color="#3ecfb2", ha="center")
plt.annotate("z << 0 → σ → 0.0
(very confident: Class 0)",
             xy=(-5, 0.05), fontsize=9, color="#e06060", ha="center")
plt.annotate("z = 0 → σ = 0.5
(uncertain)",
             xy=(1.5, 0.52), fontsize=9, color="#f0a040")

plt.fill_between(z, 0.5, sigmoid, where=(sigmoid >= 0.5),
                 alpha=0.12, color="#3ecfb2", label="Predict Class 1")
plt.fill_between(z, sigmoid, 0.5, where=(sigmoid < 0.5),
                 alpha=0.12, color="#e06060", label="Predict Class 0")

plt.title("The Sigmoid Function", fontsize=13, fontweight="bold")
plt.xlabel("z  (linear combination of features)")
plt.ylabel("σ(z)  — Probability")
plt.legend(loc="center left")
plt.tight_layout()
plt.show()

## Part 3 — Dataset 1: Student Admission Prediction

**Scenario:** A university wants to predict whether a student will be
admitted based on their entrance exam score and interview score.

**Features:**
- `exam_score` — marks in the entrance exam (0–100)
- `interview_score` — marks in the interview (0–100)

**Target:**
- `admitted` — 1 = Admitted, 0 = Rejected


In [ ]:
# ── CREATE DATASET ──────────────────────────────────────────────
n = 300

exam_score      = np.round(np.random.uniform(40, 100, n), 1)
interview_score = np.round(np.random.uniform(40, 100, n), 1)

# Admission rule: weighted sum with noise
score_combo = 0.6 * exam_score + 0.4 * interview_score
noise       = np.random.normal(0, 6, n)
admitted    = (score_combo + noise >= 72).astype(int)

df = pd.DataFrame({
    "exam_score"      : exam_score,
    "interview_score" : interview_score,
    "admitted"        : admitted
})
df["status"] = df["admitted"].map({1: "Admitted", 0: "Rejected"})

print(f"Dataset shape : {df.shape}")
print(f"\nAdmission rates:")
print(df["status"].value_counts())
print(f"\nAdmission rate: {admitted.mean()*100:.1f}%")
df.head(8)

In [ ]:
# ── EDA ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = {"Admitted": "#3ecfb2", "Rejected": "#e06060"}

# Scatter plot
sns.scatterplot(data=df, x="exam_score", y="interview_score",
                hue="status", palette=palette,
                alpha=0.7, s=50, edgecolor="white", ax=axes[0])
axes[0].set_title("Exam vs Interview Score", fontweight="bold")

# Box plots
sns.boxplot(data=df, x="status", y="exam_score",
            palette=palette, ax=axes[1])
axes[1].set_title("Exam Score by Status", fontweight="bold")

sns.boxplot(data=df, x="status", y="interview_score",
            palette=palette, ax=axes[2])
axes[2].set_title("Interview Score by Status", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation check
print("=== Mean scores by admission status ===")
print(df.groupby("status")[["exam_score","interview_score"]].mean().round(2))
print()
corr = df[["exam_score","interview_score","admitted"]].corr()
print("=== Correlation with admitted ===")
print(corr["admitted"].drop("admitted").sort_values(ascending=False))

## Part 4 — Build the Logistic Regression Model

In [ ]:
# ── PREPARE DATA ────────────────────────────────────────────────
features = ["exam_score", "interview_score"]
X = df[features]
y = df["admitted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling — Logistic Regression works better with scaled features
scaler  = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train : {X_train_sc.shape}  |  Test : {X_test_sc.shape}")
print()
print("Why scale? Logistic Regression uses gradient descent.")
print("Features on very different scales make it slow and unstable.")
print("Scaling puts all features on the same playing field.")

In [ ]:
# ── TRAIN ───────────────────────────────────────────────────────
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_sc, y_train)

print("✅ Model trained!")
print()
print("=== Learned Coefficients ===")
for feat, coef in zip(features, log_reg.coef_[0]):
    direction = "↑ increases" if coef > 0 else "↓ decreases"
    print(f"  {feat:20s}: {coef:+.4f}  → higher {feat} {direction} admission chance")
print(f"  {'intercept':20s}: {log_reg.intercept_[0]:+.4f}")
print()
print("Larger absolute value = stronger influence on the prediction.")

## Part 5 — Predictions & Probability Scores

The most powerful feature of Logistic Regression:
it gives you a **probability**, not just a class label.


In [ ]:
# ── PREDICT ─────────────────────────────────────────────────────
y_pred      = log_reg.predict(X_test_sc)
y_prob      = log_reg.predict_proba(X_test_sc)[:, 1]  # prob of class 1

results = pd.DataFrame({
    "exam_score"      : X_test["exam_score"].values,
    "interview_score" : X_test["interview_score"].values,
    "actual"          : y_test.values,
    "predicted"       : y_pred,
    "prob_admitted"   : np.round(y_prob, 3),
}).sort_values("prob_admitted", ascending=False).reset_index(drop=True)

results["actual_label"]    = results["actual"].map({1:"Admitted",0:"Rejected"})
results["predicted_label"] = results["predicted"].map({1:"Admitted",0:"Rejected"})
results["correct"]         = results["actual"] == results["predicted"]

print("Top 10 predictions (sorted by confidence):")
results[["exam_score","interview_score",
         "actual_label","predicted_label","prob_admitted","correct"]].head(10)

In [ ]:
# ── PREDICT FOR NEW STUDENTS ────────────────────────────────────
print("=== Predict for New Applicants ===\n")

new_students = pd.DataFrame({
    "exam_score"      : [55, 72, 88, 65, 91],
    "interview_score" : [60, 68, 85, 55, 78],
    "name"            : ["Arjun","Priya","Rohan","Divya","Sneha"]
})

new_scaled = scaler.transform(new_students[features])
new_prob   = log_reg.predict_proba(new_scaled)[:, 1]
new_pred   = log_reg.predict(new_scaled)

for _, row in new_students.iterrows():
    i = row.name
    label = "✅ ADMITTED" if new_pred[i] == 1 else "❌ REJECTED"
    print(f"  {row['name']:8s} | Exam:{row['exam_score']:5.1f} "
          f"Interview:{row['interview_score']:5.1f} "
          f"| Prob: {new_prob[i]:.1%}  →  {label}")

## Part 6 — Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc*100:.1f}%")
print()
print("=== Classification Report ===")
print(classification_report(y_test, y_pred,
      target_names=["Rejected", "Admitted"]))

In [ ]:
# ── CONFUSION MATRIX ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Rejected","Admitted"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix", fontsize=13, fontweight="bold")

# ── PROBABILITY DISTRIBUTION ─────────────────────────────────────
admitted_probs = y_prob[y_test == 1]
rejected_probs = y_prob[y_test == 0]

axes[1].hist(rejected_probs, bins=20, alpha=0.65,
             color="#e06060", edgecolor="white", label="Actual: Rejected")
axes[1].hist(admitted_probs, bins=20, alpha=0.65,
             color="#3ecfb2", edgecolor="white", label="Actual: Admitted")
axes[1].axvline(0.5, color="#f0a040", linewidth=2,
                linestyle="--", label="Threshold = 0.5")
axes[1].set_title("Predicted Probability Distribution",
                   fontsize=13, fontweight="bold")
axes[1].set_xlabel("P(Admitted)")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Good model = two well-separated humps in the probability distribution.")

In [ ]:
# ── ROC CURVE & AUC ─────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color="#7c6af7", linewidth=2.5,
         label=f"Logistic Regression (AUC = {auc_score:.3f})")
plt.plot([0,1],[0,1], color="white", linewidth=1,
         linestyle="--", alpha=0.4, label="Random classifier (AUC = 0.5)")
plt.fill_between(fpr, tpr, alpha=0.08, color="#7c6af7")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve", fontsize=13, fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(f"AUC = {auc_score:.3f}")
print()
print("AUC interpretation:")
print("  1.0 = perfect model")
print("  0.5 = random guessing (useless)")
print(f"  {auc_score:.2f} = {'Excellent' if auc_score>0.9 else 'Good' if auc_score>0.8 else 'Fair'}!")

In [ ]:
# ── DECISION BOUNDARY ───────────────────────────────────────────
h = 0.3
x_min, x_max = df["exam_score"].min()-2,      df["exam_score"].max()+2
y_min, y_max = df["interview_score"].min()-2, df["interview_score"].max()+2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                      np.arange(y_min, y_max, h))

grid_scaled = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
Z = log_reg.predict_proba(grid_scaled)[:, 1].reshape(xx.shape)

plt.figure(figsize=(9, 6))
contour = plt.contourf(xx, yy, Z, levels=20, cmap="RdYlGn", alpha=0.5)
plt.colorbar(contour, label="P(Admitted)")
plt.contour(xx, yy, Z, levels=[0.5], colors="#f0a040",
            linewidths=2.5, linestyles="--")

sns.scatterplot(data=df, x="exam_score", y="interview_score",
                hue="status", palette={"Admitted":"#3ecfb2","Rejected":"#e06060"},
                alpha=0.7, s=40, edgecolor="white")

plt.title("Decision Boundary — Logistic Regression", fontsize=13, fontweight="bold")
plt.xlabel("Exam Score"); plt.ylabel("Interview Score")
plt.tight_layout()
plt.show()

print("Orange dashed line = decision boundary (P = 0.5)")
print("Green region = model predicts Admitted")
print("Red region   = model predicts Rejected")

## Part 7 — Dataset 2: Diabetes Prediction

Now let's apply Logistic Regression to a **medical dataset**.

**Task:** Predict whether a patient has diabetes based on health indicators.

**Features:** Glucose, Blood Pressure, BMI, Age (4 features from the classic Pima Indians dataset)


In [ ]:
# ── CREATE MEDICAL DATASET ──────────────────────────────────────
n2 = 500

glucose     = np.round(np.random.normal(120, 30, n2)).clip(60, 200)
blood_press = np.round(np.random.normal(72, 12, n2)).clip(40, 120)
bmi         = np.round(np.random.normal(32, 7, n2), 1).clip(15, 60)
age         = np.random.randint(21, 75, n2)

# Diabetes risk increases with higher glucose, BMI, and age
risk = (0.04*glucose + 0.02*bmi + 0.015*age
        - 0.01*blood_press - 6.5
        + np.random.normal(0, 0.8, n2))
diabetes = (risk > 0).astype(int)

med_df = pd.DataFrame({
    "glucose"     : glucose,
    "blood_press" : blood_press,
    "bmi"         : bmi,
    "age"         : age,
    "diabetes"    : diabetes
})

print(f"Dataset shape: {med_df.shape}")
print(f"\nDiabetes prevalence: {diabetes.mean()*100:.1f}%")
med_df.head(6)

In [ ]:
# ── EDA ─────────────────────────────────────────────────────────
med_features = ["glucose", "blood_press", "bmi", "age"]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, feat in zip(axes.flatten(), med_features):
    for label, color, name in [(0,"#3ecfb2","No Diabetes"),
                                 (1,"#e06060","Diabetes")]:
        grp = med_df[med_df["diabetes"] == label][feat]
        ax.hist(grp, bins=20, alpha=0.6, color=color,
                edgecolor="white", label=name)
    ax.set_title(f"{feat}", fontweight="bold")
    ax.legend()

plt.suptitle("Feature Distributions — Diabetes vs No Diabetes",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── TRAIN & EVALUATE ────────────────────────────────────────────
X2 = med_df[med_features]
y2 = med_df["diabetes"]

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

sc2    = StandardScaler()
X2_trs = sc2.fit_transform(X2_tr)
X2_tes = sc2.transform(X2_te)

med_model = LogisticRegression(random_state=42)
med_model.fit(X2_trs, y2_tr)

y2_pred = med_model.predict(X2_tes)
y2_prob = med_model.predict_proba(X2_tes)[:, 1]

print(f"Diabetes Model Accuracy: {accuracy_score(y2_te, y2_pred)*100:.1f}%")
print(f"AUC Score              : {roc_auc_score(y2_te, y2_prob):.3f}")
print()
print(classification_report(y2_te, y2_pred,
      target_names=["No Diabetes","Diabetes"]))

In [ ]:
# ── FEATURE IMPORTANCE via COEFFICIENTS ─────────────────────────
coef_df = pd.DataFrame({
    "feature"    : med_features,
    "coefficient": med_model.coef_[0]
}).sort_values("coefficient")

colors = ["#e06060" if c > 0 else "#3ecfb2" for c in coef_df["coefficient"]]

plt.figure(figsize=(8, 4))
plt.barh(coef_df["feature"], coef_df["coefficient"],
         color=colors, edgecolor="white")
plt.axvline(0, color="white", linewidth=1)
plt.title("Logistic Regression Coefficients — Diabetes Model",
          fontsize=13, fontweight="bold")
plt.xlabel("Coefficient (after scaling)")
plt.tight_layout()
plt.show()

print("Red  = increases diabetes risk  (positive coefficient)")
print("Green = decreases diabetes risk (negative coefficient)")

## Part 8 — Key Concepts Summary

```
┌────────────────────────────────────────────────────────────────────┐
│              LOGISTIC REGRESSION — CONCEPT MAP                    │
├─────────────────────────┬──────────────────────────────────────────┤
│  Core idea              │  Linear equation → Sigmoid → Probability │
├─────────────────────────┼──────────────────────────────────────────┤
│  Output                 │  Probability between 0 and 1             │
├─────────────────────────┼──────────────────────────────────────────┤
│  Decision threshold     │  Usually 0.5 (adjustable!)               │
├─────────────────────────┼──────────────────────────────────────────┤
│  Needs feature scaling  │  Yes — always use StandardScaler         │
├─────────────────────────┼──────────────────────────────────────────┤
│  Strengths              │  ✅ Gives probabilities, not just labels │
│                         │  ✅ Fast to train, interpretable         │
│                         │  ✅ Works great when classes are         │
│                         │     linearly separable                   │
├─────────────────────────┼──────────────────────────────────────────┤
│  Weaknesses             │  ❌ Assumes a linear decision boundary   │
│                         │  ❌ Struggles with complex patterns      │
├─────────────────────────┼──────────────────────────────────────────┤
│  Key metrics            │  Accuracy, Precision, Recall, F1, AUC   │
└─────────────────────────┴──────────────────────────────────────────┘

  Linear Regression    →  Decision Trees  →  Logistic Regression
  (predict numbers)       (rules/splits)      (probabilities)
                                ↓
                        NEXT: Ensemble Methods
                        (Random Forest, Boosting)
```


## 🧪 Challenges

### Challenge 1 — Adjust the Threshold (Easy)
The default threshold is 0.5. Change it to **0.3** for the diabetes model.
How does this affect precision, recall, and accuracy?

*Hint: `y_pred_new = (y2_prob >= 0.3).astype(int)`*

Which threshold would you prefer for a medical model — and why?


In [ ]:
# Challenge 1 — Threshold tuning
# Try thresholds: 0.3, 0.4, 0.5, 0.6, 0.7
# Print accuracy, precision and recall for each
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    # your code here
    pass

### Challenge 2 — New Patient Prediction (Easy)
Predict the diabetes probability for these 3 patients:

| Patient | Glucose | Blood Pressure | BMI  | Age |
|---------|---------|----------------|------|-----|
| A       | 148     | 72             | 33.6 | 50  |
| B       | 85      | 66             | 26.6 | 31  |
| C       | 183     | 64             | 23.3 | 32  |

Who is at highest risk?


In [ ]:
# Challenge 2 — Predict for new patients
new_patients = pd.DataFrame({
    "glucose"     : [148, 85, 183],
    "blood_press" : [72,  66, 64 ],
    "bmi"         : [33.6, 26.6, 23.3],
    "age"         : [50,  31, 32 ]
})
# Your code here

### Challenge 3 — Add More Features (Medium)
Add two new columns to the diabetes dataset:
- `insulin` — random normal, mean=80, std=115, clipped to [0, 400]
- `skin_thickness` — random normal, mean=20, std=15, clipped to [0, 99]

Retrain the model with all 6 features. Does accuracy and AUC improve?


In [ ]:
# Challenge 3 — Add insulin and skin_thickness
np.random.seed(10)
med_df["insulin"]        = np.round(np.random.normal(80, 115, n2), 1).clip(0, 400)
med_df["skin_thickness"] = np.round(np.random.normal(20, 15,  n2), 1).clip(0, 99)

# Retrain and compare accuracy and AUC
# Your code here

### Challenge 4 — Logistic Regression vs Decision Tree (Hard)
Train **both** a Logistic Regression and a Decision Tree on the
diabetes dataset. Compare them side-by-side using:
- Accuracy
- AUC score
- ROC curves plotted on the same chart

Which model would you recommend for this medical use case?


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Train both models on the same train/test split
# Plot both ROC curves on the same figure
# Your code here

## Wrap-Up

Today you learned:

- ✅ Why Linear Regression fails for classification (outputs outside 0–1)
- ✅ The Sigmoid function — squashes any number into a probability
- ✅ How to train, interpret, and evaluate a Logistic Regression model
- ✅ Feature scaling with StandardScaler and why it matters
- ✅ Confusion matrix, classification report, probability distribution plot
- ✅ ROC curve and AUC — the gold standard for binary classifier evaluation
- ✅ Decision boundary visualisation
- ✅ Applied everything to a medical (diabetes) dataset
- ✅ How coefficients reveal which features drive predictions

### What's next
**Ensemble Methods** — Random Forests, Voting Classifiers, and Bagging.
Instead of one model, we combine many. Almost always more powerful.
